# SANN / VA-SANN / CA-SANN — 3D Growth Visualization (Plotly)

This notebook creates a **fully interactive 3D visualization** of your adaptive neural networks using **Plotly**:

- **Neurons** → 3D points (Scatter3d)
- **Connections** → 3D lines (Scatter3d)
- **Layers** along **X-axis**; neurons stacked in **Y/Z**
- **Animated over epochs** with a **slider** + **play/pause**
- **Toggles**: show/hide connections, show/hide growth events
- **Overlays**: accuracy vs epoch, model size vs epoch
- **Decision flash** per epoch: accepted → green flash, rejected → red flash

Color coding:
- Existing neurons: **blue**
- New neurons: **green**
- Removed neurons: **red** (if layer width decreases)
- Output layer: **yellow**

Data sources (required):
- `seed_results.csv` (run-level summary)
- `result.json` (per-epoch time_series + growth counters)

Note: `result.json` typically does **not** store per-layer widths per epoch. For accurate structure-per-epoch animation, this notebook will **optionally read checkpoints** if present (from the `final_checkpoint` path in `result.json`).

In [1]:
import sys
import json
import re
import importlib
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

def ensure_package(pkg: str, import_name: str | None = None) -> None:
    name = import_name or pkg
    try:
        importlib.import_module(name)
        return
    except Exception:
        pass
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('plotly')
ensure_package('nbformat')

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Torch is only needed if you want checkpoint-based structure inference
ensure_package('torch', import_name='torch')
import torch

RUNS_DIR = Path('runs')
assert RUNS_DIR.exists(), f'Missing runs directory: {RUNS_DIR.resolve()}'
print('Runs dir:', RUNS_DIR.resolve())

Installing plotly...
Installing nbformat...
Runs dir: C:\SSAN\runs


In [2]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def find_latest_run_with_seed_results(runs_dir: Path) -> Path | None:
    candidates = list(runs_dir.glob('**/seed_results.csv'))
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0].parent

def list_available_result_jsons(run_dir: Path) -> pd.DataFrame:
    rows = []
    for result_path in run_dir.glob('**/result.json'):
        parts = result_path.parts
        try:
            model = parts[-2]
            seed_folder = parts[-3]
            dataset = parts[-4]
        except Exception:
            continue
        if not seed_folder.startswith('seed_'):
            continue
        try:
            seed = int(seed_folder.split('_', 1)[1])
        except Exception:
            continue
        payload = load_json(result_path)
        rows.append({
            'dataset': dataset,
            'seed': seed,
            'model': model,
            'path': str(result_path),
            'test_accuracy': float(payload.get('test_accuracy', 0.0)),
            'model_size': int(payload.get('model_size', 0)),
            'growth_events': int(payload.get('growth_events', 0)),
            'rejected_growth_events': int(payload.get('rejected_growth_events', 0)),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['dataset', 'seed', 'model']).reset_index(drop=True)
    return df

DEFAULT_RUN_DIR = find_latest_run_with_seed_results(RUNS_DIR)
print('Default run dir:', DEFAULT_RUN_DIR)

RUN_DIR = DEFAULT_RUN_DIR if DEFAULT_RUN_DIR is not None else RUNS_DIR
print('Using RUN_DIR:', RUN_DIR)

df_runs = list_available_result_jsons(RUN_DIR)
display(df_runs.head(30))
print('Found result.json files:', len(df_runs))

Default run dir: runs\cifar10_underfit_delayed_acceptance_smoke
Using RUN_DIR: runs\cifar10_underfit_delayed_acceptance_smoke


,dataset,seed,model,path,test_accuracy,model_size,growth_events,rejected_growth_events
0,cifar10,13,CA-SANN,runs\cifar10_underfit_delayed_acceptance_smoke...,0.328125,96198,1,0
1,cifar10,13,SANN,runs\cifar10_underfit_delayed_acceptance_smoke...,0.324219,96198,1,0
2,cifar10,13,Static,runs\cifar10_underfit_delayed_acceptance_smoke...,0.250000,94986,0,0


Found result.json files: 3


In [3]:
# ----------------------------
# Choose what to visualize
# ----------------------------
# Edit these defaults if you want to point to a different run/dataset/seed/model.

# If df_runs is non-empty, pick the first row as a sensible default.
if 'df_runs' in globals() and df_runs is not None and not getattr(df_runs, 'empty', True):
    _row0 = df_runs.iloc[0]
    DATASET = str(_row0['dataset'])
    SEED = int(_row0['seed'])
    MODEL = str(_row0['model'])
else:
    DATASET = 'cifar10'
    SEED = 13
    MODEL = 'CA-SANN'

print('Selected:', {'dataset': DATASET, 'seed': SEED, 'model': MODEL})

# Required inputs
seed_results_path = RUN_DIR / 'seed_results.csv'
if not seed_results_path.exists():
    raise FileNotFoundError(f'Missing seed_results.csv at: {seed_results_path.resolve()}')
df_seed = pd.read_csv(seed_results_path)
display(df_seed.head())

result_json_path = RUN_DIR / DATASET / f'seed_{SEED}' / MODEL / 'result.json'
if not result_json_path.exists():
    raise FileNotFoundError(f'Missing result.json at: {result_json_path.resolve()}')
payload = load_json(result_json_path)
print('Loaded result.json:', result_json_path)

ts = payload.get('time_series', {})
df_ts = pd.DataFrame(ts)
if 'epoch' not in df_ts.columns or df_ts.empty:
    raise ValueError('result.json has no usable time_series/epoch')
df_ts['epoch'] = df_ts['epoch'].astype(int)
df_ts = df_ts.sort_values('epoch').reset_index(drop=True)
display(df_ts.head())

epochs = df_ts['epoch'].astype(int).to_list()
val_acc = df_ts.get('val_accuracy', pd.Series([np.nan] * len(df_ts))).astype(float).to_list()
model_size = df_ts.get('model_size', pd.Series([np.nan] * len(df_ts))).astype(float).to_list()

# Growth / decision signals (per-epoch) from cumulative counters
def _delta_events(series: pd.Series) -> list[int]:
    vals = series.fillna(0).astype(float).to_numpy()
    deltas = np.diff(np.concatenate([[0.0], vals]))
    return [int(d) for d in deltas]

accepted_delta = _delta_events(df_ts.get('accepted_growth_events', df_ts.get('growth_events', pd.Series([0] * len(df_ts)))))
rejected_delta = _delta_events(df_ts.get('rejected_growth_events', pd.Series([0] * len(df_ts))))
any_growth_delta = _delta_events(df_ts.get('growth_events', pd.Series([0] * len(df_ts))))

growth_event_epochs = [e for e, d in zip(epochs, any_growth_delta) if d > 0]
accepted_epochs = [e for e, d in zip(epochs, accepted_delta) if d > 0]
rejected_epochs = [e for e, d in zip(epochs, rejected_delta) if d > 0]
print('Growth epochs   :', growth_event_epochs)
print('Accepted epochs :', accepted_epochs)
print('Rejected epochs :', rejected_epochs)

final_ckpt = payload.get('final_checkpoint', '')
print('final_checkpoint:', final_ckpt)

Selected: {'dataset': 'cifar10', 'seed': 13, 'model': 'CA-SANN'}


,dataset,seed,model,val_accuracy,test_accuracy,val_loss,test_loss,model_size,efficiency,accuracy_per_100k_params,growth_events,candidate_growth_events,exploration_growth_events,rejected_growth_events,training_time_sec
0,cifar10,13,Static,0.253906,0.250000,2.076360,2.070488,94986,0.000003,0.263197,0,0,0,0,14.865316
1,cifar10,13,SANN,0.285156,0.324219,1.984997,1.954931,96198,0.000003,0.337033,1,1,0,0,17.249273
2,cifar10,13,CA-SANN,0.300781,0.328125,1.978946,1.960993,96198,0.000003,0.341093,1,1,0,0,17.334544


Loaded result.json: runs\cifar10_underfit_delayed_acceptance_smoke\cifar10\seed_13\CA-SANN\result.json


,epoch,val_accuracy,val_loss,model_size,efficiency,peak_efficiency,growth_events,candidate_growth_events,exploration_growth_events,accepted_growth_events,rejected_growth_events,capacity_status,difficulty_score,growth_allowed
0,1,0.203125,2.167547,96198,0.000002,0.000002,0,1,0,0,0,underfit,0.438227,1
1,2,0.250000,2.090997,96198,0.000003,0.000003,0,1,0,0,0,underfit,0.392919,1
2,3,0.281250,2.006516,96198,0.000003,0.000003,1,1,0,1,0,underfit,0.371244,1
3,4,0.300781,1.978946,96198,0.000003,0.000003,1,1,0,1,0,underfit,0.383112,1


Growth epochs   : [3]
Accepted epochs : [3]
Rejected epochs : []
final_checkpoint: runs\cifar10_underfit_delayed_acceptance_smoke\cifar10\seed_13\CA-SANN\checkpoints\ca_sann_epoch004_final.pt


In [ ]:
# ---------------------------------------------
# 3D Plotly animation: structure + metrics (HUD re-skin)
# ---------------------------------------------

# Rendering controls (adjust if your layers are large)
MAX_NEURONS_PER_HIDDEN_LAYER = 120   # downsample per layer for smooth interactivity
MAX_OUTPUT_NEURONS = 40
MAX_EDGES = 900                      # reduce clutter (futuristic + readable)
RANDOM_SEED_FOR_EDGES = 7

# Sci-fi HUD palette (controlled brightness)
COLOR_EXISTING = '#66E3FF'  # soft cyan
COLOR_NEW = '#2CFFB2'       # mint neon
COLOR_REMOVED = '#FF4D6D'   # hot coral
COLOR_OUTPUT = '#FFC857'    # warm amber

UI_BG = '#05060A'
EDGE_COLOR = 'rgba(163, 110, 255, 0.16)'      # violet plasma
EDGE_GLOW_COLOR = 'rgba(163, 110, 255, 0.06)'
EDGE_WIDTH = 1.5
EDGE_GLOW_WIDTH = 5
NODE_OUTLINE = 'rgba(255, 255, 255, 0.10)'
FRAME_COLOR = 'rgba(102, 227, 255, 0.12)'

# Glow settings (marker-only glow trick)
GLOW_SIZE = 14
GLOW_OPACITY = 0.14

@dataclass
class Snapshot:
    epoch: int
    hidden_widths: list[int]
    output_width: int


def _epoch_from_name(name: str) -> int | None:
    m = re.search(r"epoch(\d+)", name.lower())
    if not m:
        return None
    try:
        return int(m.group(1))
    except Exception:
        return None


def _checkpoint_dir_from_payload(payload: dict) -> Path | None:
    fc = payload.get('final_checkpoint')
    if not fc:
        return None
    p = Path(str(fc))
    if p.exists():
        return p.parent
    # Windows backslash normalization
    p2 = Path(str(fc).replace('\\', '/'))
    if p2.exists():
        return p2.parent
    return None


def _map_epoch_to_checkpoint(ckpt_dir: Path) -> dict[int, Path]:
    mapping: dict[int, Path] = {}
    for p in ckpt_dir.glob('*.pt'):
        e = _epoch_from_name(p.name)
        if e is None:
            continue
        if e not in mapping:
            mapping[e] = p
        else:
            # Prefer '*final*'
            if ('final' in p.name.lower()) and ('final' not in mapping[e].name.lower()):
                mapping[e] = p
    return mapping


def _state_dict_from_checkpoint(path: Path) -> dict[str, Any]:
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    state = ckpt.get('model_state_dict') or ckpt.get('model') or ckpt.get('state_dict')
    if state is None:
        raise KeyError(f'No model state dict found in checkpoint: {path}')
    return state


def _infer_hidden_widths_and_output_width(state: dict[str, Any]) -> tuple[list[int], int]:
    # CNN: hidden_layers.{i}.conv.weight
    if any('.conv.weight' in k for k in state.keys()):
        widths: list[int] = []
        idx = 0
        while True:
            key = f'hidden_layers.{idx}.conv.weight'
            if key not in state:
                break
            w = state[key]
            widths.append(int(w.shape[0]))
            idx += 1
        out_w = int(state['output_layer.weight'].shape[0]) if 'output_layer.weight' in state else 10
        return widths, out_w

    # MLP: hidden_layers.{i}.linear.weight or hidden_layers.{i}.weight
    widths = []
    idx = 0
    while True:
        key = None
        for cand in (f'hidden_layers.{idx}.linear.weight', f'hidden_layers.{idx}.weight'):
            if cand in state:
                key = cand
                break
        if key is None:
            break
        w = state[key]
        widths.append(int(w.shape[0]))
        idx += 1
    out_w = int(state['output_layer.weight'].shape[0]) if 'output_layer.weight' in state else 10
    return widths, out_w


def _snapshot_for_epoch(epoch: int, epoch_to_ckpt: dict[int, Path], last: Snapshot | None) -> Snapshot:
    # exact epoch checkpoint; else most-recent prior; else fallback
    chosen: Path | None = None
    if epoch_to_ckpt:
        if epoch in epoch_to_ckpt:
            chosen = epoch_to_ckpt[epoch]
        else:
            prior = [e for e in epoch_to_ckpt.keys() if e <= epoch]
            if prior:
                chosen = epoch_to_ckpt[max(prior)]

    if chosen is not None:
        state = _state_dict_from_checkpoint(chosen)
        widths, out_w = _infer_hidden_widths_and_output_width(state)
        return Snapshot(epoch=epoch, hidden_widths=widths, output_width=out_w)

    if last is not None:
        return Snapshot(epoch=epoch, hidden_widths=list(last.hidden_widths), output_width=int(last.output_width))

    # last resort: simple placeholder
    return Snapshot(epoch=epoch, hidden_widths=[64, 64], output_width=10)


def _sample_indices(width: int, max_units: int) -> list[int]:
    if width <= max_units:
        return list(range(width))
    if max_units <= 1:
        return [0]
    # evenly spaced deterministic sample
    return sorted({round(i * (width - 1) / (max_units - 1)) for i in range(max_units)})


def _grid_positions(n: int) -> tuple[np.ndarray, np.ndarray]:
    # returns (y, z) centered grid
    if n <= 0:
        return np.array([]), np.array([])
    rows = int(np.ceil(np.sqrt(n)))
    cols = int(np.ceil(n / rows))
    ys = []
    zs = []
    for i in range(n):
        r = i // cols
        c = i % cols
        ys.append(c)
        zs.append(-r)
    y = np.array(ys, dtype=float)
    z = np.array(zs, dtype=float)
    y -= y.mean()
    z -= z.mean()
    return y, z


def _build_nodes_for_snapshot(current: Snapshot, previous: Snapshot | None) -> dict[str, Any]:
    # returns dict with x,y,z arrays and per-node color
    xs: list[float] = []
    ys: list[float] = []
    zs: list[float] = []
    colors: list[str] = []
    layer_ids: list[int] = []

    def add_layer(layer_index: int, width: int, *, prev_width: int | None, is_output: bool) -> None:
        sampled = _sample_indices(width, MAX_OUTPUT_NEURONS if is_output else MAX_NEURONS_PER_HIDDEN_LAYER)
        y, z = _grid_positions(len(sampled))

        grew_by = 0
        shrank_by = 0
        if prev_width is not None:
            grew_by = max(0, int(width - prev_width))
            shrank_by = max(0, int(prev_width - width))

        for j, idx in enumerate(sampled):
            xs.append(float(layer_index))
            ys.append(float(y[j]))
            zs.append(float(z[j]))
            layer_ids.append(layer_index)

            if is_output:
                colors.append(COLOR_OUTPUT)
            else:
                # Mark new nodes: indices near the end are considered new when the layer grew
                if prev_width is not None and grew_by > 0 and idx >= (width - grew_by):
                    colors.append(COLOR_NEW)
                else:
                    colors.append(COLOR_EXISTING)

        # If a layer shrank, visualize removed nodes (red) as ghost points
        if (not is_output) and prev_width is not None and shrank_by > 0:
            removed_indices = _sample_indices(shrank_by, min(18, shrank_by))
            y2, z2 = _grid_positions(len(removed_indices))
            for k, _ in enumerate(removed_indices):
                xs.append(float(layer_index))
                ys.append(float(y2[k]))
                zs.append(float(z2[k]) - 0.7)
                layer_ids.append(layer_index)
                colors.append(COLOR_REMOVED)

    prev_widths = previous.hidden_widths if previous is not None else []
    for i, w in enumerate(current.hidden_widths):
        prev_w = prev_widths[i] if i < len(prev_widths) else None
        add_layer(i, int(w), prev_width=prev_w, is_output=False)

    out_layer_index = len(current.hidden_widths)
    add_layer(out_layer_index, int(current.output_width), prev_width=None, is_output=True)

    return {
        'x': np.array(xs),
        'y': np.array(ys),
        'z': np.array(zs),
        'color': colors,
        'layer_ids': np.array(layer_ids),
    }


def _build_edges(node_pack: dict[str, Any], hidden_widths: list[int]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Downsampled connections: connect sampled units between adjacent layers
    rng = np.random.default_rng(RANDOM_SEED_FOR_EDGES)
    x = node_pack['x']
    y = node_pack['y']
    z = node_pack['z']
    layer_ids = node_pack['layer_ids']
    color = np.array(node_pack['color'], dtype=object)

    # Exclude ghost removed nodes from edge endpoints
    is_real = color != COLOR_REMOVED

    indices_by_layer: dict[int, np.ndarray] = {}
    for layer in np.unique(layer_ids):
        mask = (layer_ids == layer) & is_real
        indices_by_layer[int(layer)] = np.where(mask)[0]

    xs: list[float] = []
    ys: list[float] = []
    zs: list[float] = []

    def add_segment(i0: int, i1: int) -> None:
        xs.extend([float(x[i0]), float(x[i1]), None])
        ys.extend([float(y[i0]), float(y[i1]), None])
        zs.extend([float(z[i0]), float(z[i1]), None])

    num_layers_total = len(hidden_widths) + 1  # + output
    edge_budget = int(MAX_EDGES)

    for layer in range(num_layers_total - 1):
        src = indices_by_layer.get(layer, np.array([], dtype=int))
        dst = indices_by_layer.get(layer + 1, np.array([], dtype=int))
        if src.size == 0 or dst.size == 0:
            continue

        per_src = max(1, min(3, int(edge_budget / max(1, src.size))))
        for i0 in src:
            if edge_budget <= 0:
                break
            pick = rng.choice(dst, size=min(per_src, dst.size), replace=False)
            for i1 in pick:
                add_segment(int(i0), int(i1))
                edge_budget -= 1
        if edge_budget <= 0:
            break

    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)


def _layer_frames_for_nodes(node_pack: dict[str, Any], *, pad: float = 0.85) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Outline rectangle per layer to give a sci-fi "slice" feel.
    x = node_pack['x']
    y = node_pack['y']
    z = node_pack['z']
    layer_ids = node_pack['layer_ids']
    colors = np.array(node_pack['color'], dtype=object)

    # Use only non-removed nodes to set the frame bounds
    is_real = colors != COLOR_REMOVED

    xs: list[float] = []
    ys: list[float] = []
    zs: list[float] = []

    for layer in sorted(set(int(v) for v in np.unique(layer_ids))):
        mask = (layer_ids == layer) & is_real
        idx = np.where(mask)[0]
        if idx.size < 2:
            continue

        y0 = float(np.min(y[idx]) - pad)
        y1 = float(np.max(y[idx]) + pad)
        z0 = float(np.min(z[idx]) - pad)
        z1 = float(np.max(z[idx]) + pad)
        xl = float(layer)

        # rectangle polyline (closed)
        xs.extend([xl, xl, xl, xl, xl, None])
        ys.extend([y0, y1, y1, y0, y0, None])
        zs.extend([z0, z0, z1, z1, z0, None])

    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)


def _hex_to_rgba(color: str, a: float) -> str:
    c = color.strip()
    if c.startswith('rgba('):
        try:
            head = c.split('rgba(', 1)[1].split(')', 1)[0]
            parts = [p.strip() for p in head.split(',')]
            if len(parts) >= 3:
                r, g, b = parts[:3]
                return f'rgba({r}, {g}, {b}, {a})'
        except Exception:
            return c
    if c.startswith('#') and len(c) == 7:
        r = int(c[1:3], 16)
        g = int(c[3:5], 16)
        b = int(c[5:7], 16)
        return f'rgba({r}, {g}, {b}, {a})'
    return c


def _glow_colors(colors: list[str], a: float) -> list[str]:
    return [_hex_to_rgba(c, a) for c in colors]


# Build per-epoch structure snapshots (checkpoint-driven if possible)
ckpt_dir = _checkpoint_dir_from_payload(payload)
epoch_to_ckpt: dict[int, Path] = {}
if ckpt_dir is not None and ckpt_dir.exists():
    epoch_to_ckpt = _map_epoch_to_checkpoint(ckpt_dir)
    print('Checkpoints dir:', ckpt_dir.resolve())
    print('Checkpoint epochs:', sorted(epoch_to_ckpt.keys()))
else:
    print('No checkpoint dir found; structure will use fallback widths (less accurate).')

snapshots: list[Snapshot] = []
prev: Snapshot | None = None
for e in epochs:
    s = _snapshot_for_epoch(int(e), epoch_to_ckpt, prev)
    snapshots.append(s)
    prev = s

# Precompute nodes/edges/frames per epoch for smooth animation
node_packs: list[dict[str, Any]] = []
edge_packs: list[tuple[np.ndarray, np.ndarray, np.ndarray]] = []
frame_packs: list[tuple[np.ndarray, np.ndarray, np.ndarray]] = []
for i, s in enumerate(snapshots):
    prev_s = snapshots[i - 1] if i > 0 else None
    nodes = _build_nodes_for_snapshot(s, prev_s)
    node_packs.append(nodes)
    edge_packs.append(_build_edges(nodes, s.hidden_widths))
    frame_packs.append(_layer_frames_for_nodes(nodes))


def _decision_for_epoch(e: int) -> tuple[str, str]:
    if e in accepted_epochs:
        return ('ACCEPTED', COLOR_NEW)
    if e in rejected_epochs:
        return ('REJECTED', COLOR_REMOVED)
    return ('', 'rgba(0,0,0,0)')


# Base figure with 3 subplots: 3D structure + accuracy + model size
fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{'type': 'scene', 'rowspan': 2}, {'type': 'xy'}], [None, {'type': 'xy'}]],
    column_widths=[0.62, 0.38],
    row_heights=[0.5, 0.5],
    subplot_titles=(
        '3D Network (neurons + connections)',
        'Accuracy vs epoch',
        'Model size vs epoch',
    ),
)

nodes0 = node_packs[0]
ex0, ey0, ez0 = edge_packs[0]
fx0, fy0, fz0 = frame_packs[0]

# Layer frames
fig.add_trace(
    go.Scatter3d(
        x=fx0,
        y=fy0,
        z=fz0,
        mode='lines',
        line=dict(color=FRAME_COLOR, width=3),
        name='Layer frames',
        showlegend=False,
        hoverinfo='skip',
    ),
    row=1,
    col=1,
)

# Edges: glow + core
fig.add_trace(
    go.Scatter3d(
        x=ex0,
        y=ey0,
        z=ez0,
        mode='lines',
        line=dict(color=EDGE_GLOW_COLOR, width=EDGE_GLOW_WIDTH),
        name='Connections (glow)',
        visible=True,
        showlegend=False,
        hoverinfo='skip',
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter3d(
        x=ex0,
        y=ey0,
        z=ez0,
        mode='lines',
        line=dict(color=EDGE_COLOR, width=EDGE_WIDTH),
        name='Connections',
        visible=True,
        hoverinfo='skip',
    ),
    row=1,
    col=1,
)

# Node glow underlay
fig.add_trace(
    go.Scatter3d(
        x=nodes0['x'],
        y=nodes0['y'],
        z=nodes0['z'],
        mode='markers',
        marker=dict(
            size=GLOW_SIZE,
            color=_glow_colors(list(nodes0['color']), GLOW_OPACITY),
            opacity=1.0,
        ),
        name='Neurons (glow)',
        showlegend=False,
        hoverinfo='skip',
    ),
    row=1,
    col=1,
)

# Nodes core
fig.add_trace(
    go.Scatter3d(
        x=nodes0['x'],
        y=nodes0['y'],
        z=nodes0['z'],
        mode='markers',
        marker=dict(
            size=4.6,
            color=nodes0['color'],
            opacity=0.92,
            line=dict(width=1, color=NODE_OUTLINE),
        ),
        name='Neurons',
        hovertemplate='Layer %{x}<br>Y %{y:.2f}<br>Z %{z:.2f}<extra></extra>',
    ),
    row=1,
    col=1,
)

# Accuracy curve + current marker
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=val_acc,
        mode='lines+markers',
        name='val_accuracy',
        line=dict(width=2, color=COLOR_EXISTING),
        marker=dict(size=5, line=dict(width=1, color='rgba(255,255,255,0.10)')),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[epochs[0]],
        y=[val_acc[0]],
        mode='markers',
        name='epoch (acc)',
        marker=dict(size=11, color='rgba(255,255,255,0.92)', line=dict(width=2, color='rgba(0,0,0,0.45)')),
        hovertemplate='epoch=%{x}<br>val_acc=%{y:.4f}<extra></extra>',
    ),
    row=1,
    col=2,
)

# Growth event markers (toggleable)
growth_y = [val_acc[epochs.index(e)] if e in epochs else None for e in growth_event_epochs]
fig.add_trace(
    go.Scatter(
        x=growth_event_epochs,
        y=growth_y,
        mode='markers',
        name='growth events',
        marker=dict(size=12, symbol='x', color='rgba(255, 166, 0, 0.92)', line=dict(width=1, color='rgba(0,0,0,0.35)')),
        visible=True,
        hovertemplate='growth @ epoch=%{x}<extra></extra>',
    ),
    row=1,
    col=2,
)

# Decision text (flash)
fig.add_trace(
    go.Scatter(
        x=[epochs[0]],
        y=[val_acc[0] if val_acc[0] == val_acc[0] else 0.0],
        mode='text',
        text=[''],
        textfont=dict(color='rgba(0,0,0,0)', size=18),
        name='decision',
        showlegend=False,
        hoverinfo='skip',
    ),
    row=1,
    col=2,
)

# Model size curve + marker
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=model_size,
        mode='lines+markers',
        name='model_size',
        line=dict(width=2, color=COLOR_REMOVED),
        marker=dict(size=5, line=dict(width=1, color='rgba(255,255,255,0.10)')),
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[epochs[0]],
        y=[model_size[0]],
        mode='markers',
        name='epoch (size)',
        marker=dict(size=11, color='rgba(255,255,255,0.92)', line=dict(width=2, color='rgba(0,0,0,0.45)')),
        hovertemplate='epoch=%{x}<br>params=%{y:.0f}<extra></extra>',
    ),
    row=2,
    col=2,
)

# Axes cosmetics
fig.update_yaxes(range=[0, 1.0], row=1, col=2)
fig.update_xaxes(title_text='epoch', row=1, col=2)
fig.update_xaxes(title_text='epoch', row=2, col=2)
fig.update_yaxes(title_text='val_accuracy', row=1, col=2)
fig.update_yaxes(title_text='params', row=2, col=2)

# 3D scene cosmetics (clean HUD look)
fig.update_scenes(
    xaxis=dict(title='', showbackground=False, showgrid=False, zeroline=False, showticklabels=False, ticks=''),
    yaxis=dict(title='', showbackground=False, showgrid=False, zeroline=False, showticklabels=False, ticks=''),
    zaxis=dict(title='', showbackground=False, showgrid=False, zeroline=False, showticklabels=False, ticks=''),
    aspectmode='data',
    camera=dict(eye=dict(x=1.65, y=1.15, z=0.95)),
    bgcolor=UI_BG,
)


# Animation frames (update specific traces by index)
# Trace order now:
# 0 layer frames (3D)
# 1 edges glow (3D)
# 2 edges core (3D)
# 3 nodes glow (3D)
# 4 nodes core (3D)
# 5 acc curve
# 6 acc marker
# 7 growth markers
# 8 decision text
# 9 size curve
# 10 size marker
frames = []
for i, e in enumerate(epochs):
    nodes = node_packs[i]
    ex, ey, ez = edge_packs[i]
    fx, fy, fz = frame_packs[i]

    dec_text, dec_color = _decision_for_epoch(int(e))

    # Decision flash: tint non-output, non-removed nodes for one frame
    node_colors = list(nodes['color'])
    if dec_text == 'ACCEPTED':
        node_colors = [
            (COLOR_OUTPUT if c == COLOR_OUTPUT else (COLOR_REMOVED if c == COLOR_REMOVED else COLOR_NEW))
            for c in node_colors
        ]
    elif dec_text == 'REJECTED':
        node_colors = [
            (COLOR_OUTPUT if c == COLOR_OUTPUT else (COLOR_REMOVED if c == COLOR_REMOVED else COLOR_REMOVED))
            for c in node_colors
        ]

    frames.append(
        go.Frame(
            name=str(e),
            data=[
                go.Scatter3d(x=fx, y=fy, z=fz),
                go.Scatter3d(x=ex, y=ey, z=ez),
                go.Scatter3d(x=ex, y=ey, z=ez),
                go.Scatter3d(
                    x=nodes['x'],
                    y=nodes['y'],
                    z=nodes['z'],
                    marker=dict(color=_glow_colors(node_colors, GLOW_OPACITY)),
                ),
                go.Scatter3d(
                    x=nodes['x'],
                    y=nodes['y'],
                    z=nodes['z'],
                    marker=dict(color=node_colors),
                ),
                go.Scatter(x=[e], y=[val_acc[i]]),
                go.Scatter(x=[e], y=[val_acc[i]], text=[dec_text], textfont=dict(color=dec_color, size=18)),
                go.Scatter(x=[e], y=[model_size[i]]),
            ],
            traces=[0, 1, 2, 3, 4, 6, 8, 10],
            layout=go.Layout(title_text=f"{MODEL} — {DATASET} seed={SEED} | epoch={e} | decision={dec_text or '—'}"),
        )
    )

fig.frames = frames

# Slider for epochs
slider_steps = []
for e in epochs:
    slider_steps.append({
        'method': 'animate',
        'label': str(e),
        'args': [
            [str(e)],
            {
                'mode': 'immediate',
                'frame': {'duration': 0, 'redraw': True},
                'transition': {'duration': 150},
            }
        ],
    })

sliders = [{
    'active': 0,
    'currentvalue': {'prefix': 'Epoch: '},
    'pad': {'t': 40, 'b': 10},
    'steps': slider_steps,
}]

# UI controls: play/pause + toggles
play_pause = [
    {
        'label': 'Play',
        'method': 'animate',
        'args': [None, {
            'frame': {'duration': 450, 'redraw': True},
            'fromcurrent': True,
            'transition': {'duration': 250, 'easing': 'cubic-in-out'},
        }],
    },
    {
        'label': 'Pause',
        'method': 'animate',
        'args': [[None], {
            'frame': {'duration': 0, 'redraw': False},
            'mode': 'immediate',
            'transition': {'duration': 0},
        }],
    },
]

# Connections toggle affects BOTH edge traces (glow + core)
toggle_connections = [
    {'label': 'Connections: ON', 'method': 'restyle', 'args': [{'visible': True}, [1, 2]]},
    {'label': 'Connections: OFF', 'method': 'restyle', 'args': [{'visible': False}, [1, 2]]},
]

toggle_growth = [
    {'label': 'Growth events: ON', 'method': 'restyle', 'args': [{'visible': True}, [7]]},
    {'label': 'Growth events: OFF', 'method': 'restyle', 'args': [{'visible': False}, [7]]},
]

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=UI_BG,
    plot_bgcolor=UI_BG,
    height=760,
    width=1250,
    title_text=f"{MODEL} — {DATASET} seed={SEED} | epoch={epochs[0]}",
    font=dict(color='rgba(235, 245, 255, 0.92)'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0.02,
        bgcolor='rgba(0,0,0,0)',
    ),
    margin=dict(l=10, r=10, t=75, b=10),
    sliders=sliders,
    updatemenus=[
        {
            'type': 'buttons',
            'direction': 'left',
            'x': 0.02,
            'y': 1.08,
            'showactive': False,
            'buttons': play_pause,
        },
        {
            'type': 'buttons',
            'direction': 'left',
            'x': 0.42,
            'y': 1.08,
            'showactive': True,
            'buttons': toggle_connections,
        },
        {
            'type': 'buttons',
            'direction': 'left',
            'x': 0.68,
            'y': 1.08,
            'showactive': True,
            'buttons': toggle_growth,
        },
    ],
)

fig.show(config={'displaylogo': False, 'scrollZoom': True})

Checkpoints dir: C:\SSAN\runs\cifar10_underfit_delayed_acceptance_smoke\cifar10\seed_13\CA-SANN\checkpoints
Checkpoint epochs: [1, 4]


Checkpoints dir: C:\SSAN\runs\cifar10_underfit_delayed_acceptance_smoke\cifar10\seed_13\CA-SANN\checkpoints
Checkpoint epochs: [1, 4]


## Notes / Tips

- If the 3D figure feels slow, reduce `MAX_NEURONS_PER_HIDDEN_LAYER` and/or `MAX_EDGES` in Cell 6.
- If your run has only a few checkpoints, structure will still animate by using the most recent checkpoint at or before each epoch.
- Removed neurons (red) will only appear if a layer width decreases between epochs (most runs only grow).
- Decision flash uses `accepted_growth_events` / `rejected_growth_events` if present in `result.json -> time_series`, otherwise falls back to `growth_events` vs `rejected_growth_events`.